In [ ]:
import xgboost as xgb 
import pandas as pd 
import sklearn as sk 
import re 
import matplotlib.pyplot as plt
import seaborn as sns 
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import KFold 
from sklearn.model_selection import GridSearchCV 
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingGridSearchCV
from sklearn.metrics import accuracy_score 
from scipy.linalg import sqrtm
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.metrics import confusion_matrix
%matplotlib inline
from matplotlib.collections import LineCollection
from sklearn import metrics

In [ ]:
col = ['Photon_PT_0', 'photon_energy',
      'Muon_PT_0', 'lead_muon_energy', 
      'Muon_PT_1','nextlead_muon_energy',
      'deltar_mu_0_mu_1', 'deltar_mu_1_gamma', 
        'dimuon_mass'
             ]
def readcsv(file_path, col = col):
    # 读取CSV文件
    data = pd.read_csv(file_path, usecols=col)
    # 确保列顺序与col一致
    data = data[col]
    
    return data

In [ ]:
a10 = readcsv('./predict_data/select_data/m_a = 10 GeV.csv')
a30 = readcsv('./predict_data/select_data/m_a = 30 GeV.csv')
a50 = readcsv('./predict_data/select_data/m_a = 50 GeV.csv')
a70 = readcsv('./predict_data/select_data/m_a = 70 GeV.csv')
a85 = readcsv('./predict_data/select_data/m_a = 85 GeV.csv')
amumu = readcsv('./predict_data/select_data/gammamumu.csv')

In [ ]:
all_data = pd.concat([a10,a30,a50,a70,a85,amumu], axis = 0)
a10.shape

In [ ]:
X = all_data[['Photon_PT_0', 'photon_energy',
      'Muon_PT_0', 'lead_muon_energy', 
      'Muon_PT_1','nextlead_muon_energy',
      'deltar_mu_0_mu_1', 'deltar_mu_1_gamma', 
        'dimuon_mass'
             ]]
#Y = all_data['label']

# 倒入模型

In [ ]:
model = xgb.XGBClassifier()
model.load_model('./clf_ALP_18_300')

In [ ]:
a10.columns

In [ ]:
a10_pre = model.predict_proba(a10)[:,1:2]
a30_pre = model.predict_proba(a30)[:,1:2]
a50_pre = model.predict_proba(a50)[:,1:2]
a70_pre = model.predict_proba(a70)[:,1:2]
a85_pre = model.predict_proba(a85)[:,1:2]
amumu_pre = model.predict_proba(amumu)[:,1:2]

In [ ]:
a10_pre

In [ ]:
eff_a10 = np.array([p for p in a10_pre])
eff_a30 = np.array([p for p in a30_pre])
eff_a50 = np.array([p for p in a50_pre])
eff_a70 = np.array([p for p in a70_pre])
eff_a85 = np.array([p for p in a85_pre])
eff_amumu = np.array([p for p in amumu_pre])

In [ ]:
custom_cs_list = [48.436, 35.622, 17.185, 3.4839, 0.10672, 87]
BR_mu = [0.000203407,0.000100342,0.0000842,0.0000414,0.000017]
pb_fb = 10**(3)

cross_section_a10 = custom_cs_list[0]*pb_fb*BR_mu[0]
cross_section_a30 = custom_cs_list[1]*pb_fb*BR_mu[1]
cross_section_a50 = custom_cs_list[2]*pb_fb*BR_mu[2]
cross_section_a70 = custom_cs_list[3]*pb_fb*BR_mu[3]
cross_section_a85 = custom_cs_list[4]*pb_fb*BR_mu[4]
cross_section_amumu = custom_cs_list[5]*pb_fb

L = 150_000  # fb^{-1} = 150 ab^{-1}

import math
import numpy as np

def poisson_logcdf(n_obs, mean, rtol=1e-15):
    """Return log P(N <= n_obs | mean) using stable Poisson recurrences."""
    mean = float(mean)
    if mean < 0:
        raise ValueError("Poisson mean must be non-negative")
    if not isinstance(n_obs, (int, np.integer)) or n_obs < 0:
        raise ValueError("n_obs must be a non-negative integer")
    if mean == 0:
        return 0.0

    if n_obs < mean:
        log_p = -mean + n_obs * math.log(mean) - math.lgamma(n_obs + 1)
        relative_sum = 1.0
        term = 1.0
        n = n_obs
        while n > 0:
            term *= n / mean
            relative_sum += term
            n -= 1
            if term <= rtol * relative_sum:
                break
        return min(0.0, log_p + math.log(relative_sum))

    n = n_obs + 1
    log_p = -mean + n * math.log(mean) - math.lgamma(n + 1)
    relative_sum = 1.0
    term = 1.0
    while True:
        n += 1
        term *= mean / n
        relative_sum += term
        if term <= rtol * relative_sum:
            break
    upper_tail = math.exp(log_p + math.log(relative_sum))
    upper_tail = min(max(upper_tail, 0.0), 1.0)
    return -math.inf if upper_tail == 1.0 else math.log1p(-upper_tail)

def poisson_cls(s, b, n_obs, log_cl_b=None):
    """Exact one-channel Poisson CL_s without nuisance parameters."""
    s = float(s)
    b = float(b)
    if s < 0 or b < 0:
        raise ValueError("s and b must be non-negative")
    if log_cl_b is None:
        log_cl_b = poisson_logcdf(n_obs, b)
    log_cl_s_plus_b = poisson_logcdf(n_obs, s + b)
    log_cls = min(0.0, log_cl_s_plus_b - log_cl_b)
    return 0.0 if log_cls < -745.0 else math.exp(log_cls)


In [ ]:
# 计算权重数
norm_factor_a10 = 1.0 / len(a10_pre)
norm_factor_a30 = 1.0 / len(a30_pre)
norm_factor_a50 = 1.0 / len(a50_pre)
norm_factor_a70 = 1.0 / len(a70_pre)
norm_factor_a85 = 1.0 / len(a85_pre)
norm_factor_amumu = 1.0 / len(amumu_pre)

weights_a10 = np.ones_like(a10_pre) * cross_section_a10 * norm_factor_a10
weights_a30 = np.ones_like(a30_pre) * cross_section_a30 * norm_factor_a30
weights_a50 = np.ones_like(a50_pre) * cross_section_a50 * norm_factor_a50
weights_a70 = np.ones_like(a70_pre) * cross_section_a70 * norm_factor_a70
weights_a85 = np.ones_like(a85_pre) * cross_section_a85 * norm_factor_a85
weights_amumu = np.ones_like(amumu_pre) * cross_section_amumu * norm_factor_amumu

plt.figure(figsize=(5,3))
plt.hist(a10_pre, bins=50, color='black', histtype='step', fill=False, label=r'$m_{a} = 10$ GeV', weights=weights_a10)
plt.hist(a30_pre, bins=50, color='blue', histtype='step', fill=False, label=r'$m_{a} = 30$ GeV', weights=weights_a30)
plt.hist(a50_pre, bins=50, color='red', histtype='step', fill=False, label=r'$m_{a} = 50$ GeV', weights=weights_a50)
plt.hist(a70_pre, bins=50, color='green', histtype='step', fill=False, label=r'$m_{a} = 70$ GeV', weights=weights_a70)
plt.hist(a85_pre, bins=50, color='magenta', histtype='step', fill=False, label=r'$m_{a} = 85$ GeV', weights=weights_a85)

plt.hist(amumu_pre, bins=50, histtype='step', color='darkorange', fill=False,label='BG', weights=weights_amumu, linestyle='--')

plt.legend(ncol=2)
plt.yscale('log')
plt.xlabel('XGBoost estimator')
plt.ylabel('Weighted Normalized Events')
plt.tight_layout()
plt.savefig(f"Weighted Normalized Events of OS LOG.png")
plt.show()

In [ ]:
plt.figure(figsize=(5,3))
plt.hist(a10_pre, bins=50, color='black', histtype='step', fill=False, label=r'$m_{a} = 10$ GeV')
plt.hist(a30_pre, bins=50, color='blue', histtype='step', fill=False, label=r'$m_{a} = 30$ GeV')
plt.hist(a50_pre, bins=50, color='red', histtype='step', fill=False, label=r'$m_{a} = 50$ GeV')
plt.hist(a70_pre, bins=50, color='green', histtype='step', fill=False, label=r'$m_{a} = 70$ GeV')
plt.hist(a85_pre, bins=50, color='magenta', histtype='step', fill=False, label=r'$m_{a} = 85$ GeV')

plt.hist(amumu_pre, bins=50, histtype='step', color='darkorange', fill=False,label='BG', linestyle='--', weights = np.full_like(amumu_pre, 0.5))

plt.legend(ncol=2)
plt.yscale('log')
plt.xlabel('XGBoost estimator')
plt.ylabel('Weighted Normalized Events')
plt.tight_layout()
plt.savefig(f"Weighted Normalized Events of OS LOG.png")
plt.show()

In [ ]:
import numpy as np
# 定义阈值范围
thresholds = np.linspace(min(min(a10_pre), min(a30_pre), min(a50_pre), min(a70_pre), min(a85_pre)),
                         max(max(a10_pre), max(a30_pre), max(a50_pre), max(a70_pre), max(a85_pre)), 100)
L = 150_000  # fb^{-1} = 150 ab^{-1}
# 初始化列表
cls_values_10 = []
cls_values_30 = []
cls_values_50 = []
cls_values_70 = []
cls_values_85 = []
n_obs_values = []
eff_signal_10 = []
eff_signal_30 = []
eff_signal_50 = []
eff_signal_70 = []
eff_signal_85 = []
eff_background = []

for th in thresholds:
    # 计算信号 S_1 (a10)
    N_signal1_total = len(a10_pre)
    N_signal1_right = np.sum(a10_pre > th)
    sigma_signal1 = cross_section_a10  # 请根据具体的 cross section 填充

    S1 = sigma_signal1 * L * (N_signal1_right / N_signal1_total)
    eff_signal_10.append(N_signal1_right / N_signal1_total)

    # 计算信号 S_2 (a30)
    N_signal2_total = len(a30_pre)
    N_signal2_right = np.sum(a30_pre > th)
    sigma_signal2 = cross_section_a30  # 请根据具体的 cross section 填充

    S2 = sigma_signal2 * L * (N_signal2_right / N_signal2_total)
    eff_signal_30.append(N_signal2_right / N_signal2_total)

    # 计算信号 S_3 (a50)
    N_signal3_total = len(a50_pre)
    N_signal3_right = np.sum(a50_pre > th)
    sigma_signal3 = cross_section_a50  # 请根据具体的 cross section 填充

    S3 = sigma_signal3 * L * (N_signal3_right / N_signal3_total)
    eff_signal_50.append(N_signal3_right / N_signal3_total)

    # 计算信号 S_4 (a70)
    N_signal4_total = len(a70_pre)
    N_signal4_right = np.sum(a70_pre > th)
    sigma_signal4 = cross_section_a70  # 请根据具体的 cross section 填充

    S4 = sigma_signal4 * L * (N_signal4_right / N_signal4_total)
    eff_signal_70.append(N_signal4_right / N_signal4_total)

    # 计算信号 S_5 (a85)
    N_signal5_total = len(a85_pre)
    N_signal5_right = np.sum(a85_pre > th)
    sigma_signal5 =  cross_section_a85  # 请根据具体的 cross section 填充

    S5 = sigma_signal5 * L * (N_signal5_right / N_signal5_total)
    eff_signal_85.append(N_signal5_right / N_signal5_total)

    # 计算背景 B (amumu)
    N_background_total = len(amumu_pre)
    N_background_right = np.sum(amumu_pre > th)
    sigma_background = cross_section_amumu  # 请根据具体的 cross section 填充

    B = sigma_background * L * (N_background_right / N_background_total)

    eff_background_right = N_background_right / N_background_total
    eff_background.append(eff_background_right)

    # 单一 counting channel、无 nuisance parameter 的精确 Poisson CL_s
    # notebook 中没有真实观测数据，因此采用最接近 B 的整数背景伪观测。
    n_obs = int(np.floor(B + 0.5))
    n_obs_values.append(n_obs)
    log_cl_b = poisson_logcdf(n_obs, B)

    cls_values_10.append(poisson_cls(S1, B, n_obs, log_cl_b))
    cls_values_30.append(poisson_cls(S2, B, n_obs, log_cl_b))
    cls_values_50.append(poisson_cls(S3, B, n_obs, log_cl_b))
    cls_values_70.append(poisson_cls(S4, B, n_obs, log_cl_b))
    cls_values_85.append(poisson_cls(S5, B, n_obs, log_cl_b))

# CL_s 越小，排除能力越强；为每个信号寻找最小 CL_s。
min_CLs_10 = np.nanmin(cls_values_10)
optimal_threshold_10_idx = np.nanargmin(cls_values_10)
optimal_threshold_10 = thresholds[optimal_threshold_10_idx].item()

min_CLs_30 = np.nanmin(cls_values_30)
optimal_threshold_30_idx = np.nanargmin(cls_values_30)
optimal_threshold_30 = thresholds[optimal_threshold_30_idx].item()

min_CLs_50 = np.nanmin(cls_values_50)
optimal_threshold_50_idx = np.nanargmin(cls_values_50)
optimal_threshold_50 = thresholds[optimal_threshold_50_idx].item()

min_CLs_70 = np.nanmin(cls_values_70)
optimal_threshold_70_idx = np.nanargmin(cls_values_70)
optimal_threshold_70 = thresholds[optimal_threshold_70_idx].item()

min_CLs_85 = np.nanmin(cls_values_85)
optimal_threshold_85_idx = np.nanargmin(cls_values_85)
optimal_threshold_85 = thresholds[optimal_threshold_85_idx].item()
# 计算每个数据集的权重，用于归一化
weights_a10 = np.ones_like(a10_pre) / len(a10)
weights_a30 = np.ones_like(a30_pre) / len(a30)
weights_a50 = np.ones_like(a50_pre) / len(a50)
weights_a70 = np.ones_like(a70_pre) / len(a70)
weights_a85 = np.ones_like(a85_pre) / len(a85)
weights_amumu = np.ones_like(amumu_pre) / len(amumu)

In [ ]:
# 画图
fig, (ax_main, ax_sub, ax_eff) = plt.subplots(3, 1, figsize=(5, 8), gridspec_kw={'height_ratios': [1, 1, 1]}, sharex=True)

# 原图 (主图)
ax_main.hist(a10_pre, bins=50, color='black', histtype='step', fill=False, label=r'$m_{a} = 10$ GeV', weights=weights_a10)
ax_main.hist(a30_pre, bins=50, color='red', histtype='step', fill=False, label=r'$m_{a} = 30$ GeV', weights=weights_a30)
ax_main.hist(a50_pre, bins=50, color='blue', histtype='step', fill=False, label=r'$m_{a} = 50$ GeV', weights=weights_a50)
ax_main.hist(a70_pre, bins=50, color='green', histtype='step', fill=False, label=r'$m_{a} = 70$ GeV', weights=weights_a70)
ax_main.hist(a85_pre, bins=50, color='magenta', histtype='step', fill=False, label=r'$m_{a} = 85$ GeV', weights=weights_a85)
ax_main.hist(amumu_pre, bins=50, histtype='step', color='darkorange', fill=False, label=r'$\gamma\mu\mu$', linestyle='--', linewidth=1, weights=weights_amumu)

ax_main.set_yscale('linear')
ax_main.set_ylabel('Normalized Events', fontsize=15)
ax_main.legend(ncol=2)

# 使用 -log10(CL_s) 将最优排除点显示为向上的峰。
strength_floor = 1e-12

def exclusion_strength(cls_values):
    cls_array = np.clip(np.asarray(cls_values, dtype=float), 1e-300, 1.0)
    return np.maximum(-np.log10(cls_array), strength_floor)

# 仅对 85 GeV 的显示曲线做稳健平滑；原始 CL_s 数值保持不变。
smooth_window_85 = 11

def smooth_log_curve_for_plot(values, window=11):
    values = np.maximum(np.asarray(values, dtype=float), strength_floor)
    if values.size < 3:
        return values.copy()

    window = max(3, min(int(window), values.size))
    if window % 2 == 0:
        window -= 1
    half_window = window // 2

    # 在当前对数 y 轴对应的 log10 空间中去除单点尖刺。
    log_values = np.log10(values)
    padded = np.pad(log_values, half_window, mode='edge')
    median_log = np.array([
        np.median(padded[i:i + window]) for i in range(values.size)
    ])

    # Hann 加权平均使剩余趋势连续，同时减小窗口边缘的突变。
    kernel = np.hanning(window)
    kernel /= kernel.sum()
    median_padded = np.pad(median_log, half_window, mode='edge')
    smoothed_log = np.convolve(median_padded, kernel, mode='valid')
    return 10.0 ** smoothed_log

ax_sub.plot(thresholds, exclusion_strength(cls_values_10), color='black', label=r'$m_{a} = 10$ GeV')
ax_sub.plot(thresholds, exclusion_strength(cls_values_30), color='red', label=r'$m_{a} = 30$ GeV')
ax_sub.plot(thresholds, exclusion_strength(cls_values_50), color='blue', label=r'$m_{a} = 50$ GeV')
ax_sub.plot(thresholds, exclusion_strength(cls_values_70), color='green', label=r'$m_{a} = 70$ GeV')
cls_85_for_plot = smooth_log_curve_for_plot(
    exclusion_strength(cls_values_85), window=smooth_window_85
)
ax_sub.plot(thresholds, cls_85_for_plot, color='magenta', label=r'$m_{a} = 85$ GeV')

ax_sub.set_ylabel(r'$-\log_{10}(CL_s)$', fontsize=15)
ax_sub.grid(False)
ax_sub.set_yscale('log')
ax_sub.legend(ncol=2)
ax_sub.text(
    0.5, 0.55,
    r'$\Lambda=1~\mathrm{TeV}$' + '\n'
    r'$\mathcal{L}=150~\mathrm{ab}^{-1}$',
    transform=ax_sub.transAxes,
    ha='left',
    va='top',
    fontsize=12
)

# 鉴别效率图 (子图)
ax_eff.plot(thresholds, eff_signal_10, color='black', linestyle='-', label=r'$m_{a} = 10$ GeV')
ax_eff.plot(thresholds, eff_signal_30, color='red', linestyle='-', label=r'$m_{a} = 30$ GeV')
ax_eff.plot(thresholds, eff_signal_50, color='blue', linestyle='-', label=r'$m_{a} = 50$ GeV')
ax_eff.plot(thresholds, eff_signal_70, color='green', linestyle='-', label=r'$m_{a} = 70$ GeV')
ax_eff.plot(thresholds, eff_signal_85, color='magenta', linestyle='-', label=r'$m_{a} = 85$ GeV')
ax_eff.plot(thresholds, eff_background, color='darkorange', linestyle='--', label=r'$\gamma\mu\mu$')

ax_eff.set_xlabel('Threshold', fontsize=15)
ax_eff.set_ylabel('Efficiency', fontsize=15)
ax_eff.grid(False)
ax_eff.legend(ncol=2)

# 调整每个子图的 X 和 Y 轴刻度字体大小
ax_main.tick_params(axis='x', labelsize=12)
ax_main.tick_params(axis='y', labelsize=12)
ax_sub.tick_params(axis='x', labelsize=12)
ax_sub.tick_params(axis='y', labelsize=12)
ax_eff.tick_params(axis='x', labelsize=12)
ax_eff.tick_params(axis='y', labelsize=12)

# 调整布局并保存
plt.tight_layout()
plt.savefig("Normalized_Events_with_CLs_and_Efficiency_ML.png")
plt.show()
 

In [ ]:
import csv
import os
import numpy as np

L = 150_000  # fb^{-1} = 150 ab^{-1}

# 创建保存目录
output_dir = "ML_output"
os.makedirs(output_dir, exist_ok=True)

# 使用与显著度单元相同的一维阈值范围
all_signal_scores = np.concatenate([
    np.ravel(a10_pre),
    np.ravel(a30_pre),
    np.ravel(a50_pre),
    np.ravel(a70_pre),
    np.ravel(a85_pre)
])
thresholds = np.linspace(all_signal_scores.min(), all_signal_scores.max(), 100)

# 初始化效率与逐阈值 CL_s 列表
eff_signal_10 = []
eff_signal_30 = []
eff_signal_50 = []
eff_signal_70 = []
eff_signal_85 = []
eff_background = []
cls_values_10 = []
cls_values_30 = []
cls_values_50 = []
cls_values_70 = []
cls_values_85 = []
n_obs_values = []

for th in thresholds:
    signal_eff_1 = np.sum(a10_pre > th) / len(a10_pre)
    S1 = cross_section_a10 * L * signal_eff_1
    eff_signal_10.append(signal_eff_1)

    signal_eff_2 = np.sum(a30_pre > th) / len(a30_pre)
    S2 = cross_section_a30 * L * signal_eff_2
    eff_signal_30.append(signal_eff_2)

    signal_eff_3 = np.sum(a50_pre > th) / len(a50_pre)
    S3 = cross_section_a50 * L * signal_eff_3
    eff_signal_50.append(signal_eff_3)

    signal_eff_4 = np.sum(a70_pre > th) / len(a70_pre)
    S4 = cross_section_a70 * L * signal_eff_4
    eff_signal_70.append(signal_eff_4)

    signal_eff_5 = np.sum(a85_pre > th) / len(a85_pre)
    S5 = cross_section_a85 * L * signal_eff_5
    eff_signal_85.append(signal_eff_5)

    background_eff = np.sum(amumu_pre > th) / len(amumu_pre)
    B = cross_section_amumu * L * background_eff
    eff_background.append(background_eff)

    n_obs = int(np.floor(B + 0.5))
    n_obs_values.append(n_obs)
    log_cl_b = poisson_logcdf(n_obs, B)
    cls_values_10.append(poisson_cls(S1, B, n_obs, log_cl_b))
    cls_values_30.append(poisson_cls(S2, B, n_obs, log_cl_b))
    cls_values_50.append(poisson_cls(S3, B, n_obs, log_cl_b))
    cls_values_70.append(poisson_cls(S4, B, n_obs, log_cl_b))
    cls_values_85.append(poisson_cls(S5, B, n_obs, log_cl_b))

# 将效率和每个信号对应的 CL_s 写入 CSV
with open("results.csv", "w", newline="") as csvfile:
    csv_writer = csv.writer(csvfile)
    csv_writer.writerow([
        "Threshold",
        "Signal_Efficiency_10",
        "Signal_Efficiency_30",
        "Signal_Efficiency_50",
        "Signal_Efficiency_70",
        "Signal_Efficiency_85",
        "Background_Efficiency",
        "CLs_10",
        "CLs_30",
        "CLs_50",
        "CLs_70",
        "CLs_85"
    ])

    for i in range(len(thresholds)):
        csv_writer.writerow([
            float(thresholds[i]),
            float(eff_signal_10[i]),
            float(eff_signal_30[i]),
            float(eff_signal_50[i]),
            float(eff_signal_70[i]),
            float(eff_signal_85[i]),
            float(eff_background[i]),
            float(cls_values_10[i]),
            float(cls_values_30[i]),
            float(cls_values_50[i]),
            float(cls_values_70[i]),
            float(cls_values_85[i])
        ])

print("所有数据已写入 results.csv 文件。")
